In [ ]:
# from owner of this dataset

# About this Dataset
# Context
# This is a pre-crawled dataset, taken as subset of a bigger dataset (more than 4.7 million job listings) that was created by extracting data from Monster.com, a leading job board.

# Content
# This dataset has following fields:

# country
# country_code
# date_added
# has_expired - Always false.
# job_description - The primary field for this dataset, containing the bulk of the information on what the job is about.
# job_title
# job_type - The type of tasks and skills involved in the job. For example, "management".
# location
# organization
# page_url
# salary
# sector - The industry sector the job is in. For example, "Medical services".
# Acknowledgements
# This dataset was created by PromptCloud's in-house web-crawling service.

# Inspiration
# What kinds of jobs titles correspond with what kinds of wages?
# What can you learn about the Moster.com-based US job market based on analyzing the contents of the job descriptions?
# How do job descriptions different between different industry sectors?

In [ ]:
# import tools and configs

import polars as pl
import plotly.express as px
import datetime

pl.Config(fmt_str_lengths=100, tbl_width_chars=200)

In [ ]:
# Create Dataframe and having the first glace at the dataset.

df = pl.read_csv(r"C:\Users\Natth\Downloads\Neng_Data_Pipeline\dirty_data\monster\monster_com-job_sample.csv")



# df.schema

# df.filter(df['date_added'] != "").head()

In [ ]:
# remove string in date_added.
# remove has_expired column because it only has "No".

df = df.drop(["has_expired", "date_added"])



In [ ]:
# because salary column is based on many formats Ex. hourly, yearly, even just text, so I will treat only numeric salary and change into yearly rates without bonuses (as they are the majority).

# Moving salary column to see it easier.
cols = [c for c in df.columns if c != "salary"]
cols.insert(3, "salary")
df = df.select(cols)

# Lowering case everything.
df = df.with_columns(
    pl.col("salary").str.to_lowercase()
)

# removes spaces at start and end, $, and remaining space.
df = df.with_columns([
    pl.col("salary")
    .str.strip_chars()  
    .str.replace(r"^\$", "")
    .str.strip_chars()
    .alias("salary")
])

# Creating min and max salary columns.
df = df.with_columns([
        pl.col("salary").str.extract(r"([\d,]+\.?\d*)\s*-", 1).alias("started_from"),
        pl.col("salary").str.extract(r"-\s*([\d,]+\.?\d*)", 1).alias("up_to")
])

# remove "," in numbers to make them as float.
df = df.with_columns([
    pl.col("started_from").str.replace_all(",", "").cast(pl.Float64),
    pl.col("up_to").str.replace_all(",", "").cast(pl.Float64)
])

In [ ]:

# Yearly salary converting
df_yearly = (
    df.filter(
        pl.col("salary").str.contains("year")
    ))

# some rows have small number assumedly to be hourly, I will take them out of df_yearly.
df_yearly = (df_yearly.filter(pl.col('up_to') > 1000))

#df_yearly.filter(pl.col("salary").is_not_null()).select(["salary","started_from", "up_to"]).unique()

In [ ]:
# Monthly salary converting
df_monthly = (
    df.filter([
        pl.col("salary").str.contains("month")
    ]))

# some rows have small number assumedly to be hourly, and mistaken yearly salary. I will take them out of df_monthly.
df_monthly = (df_monthly.filter([pl.col('up_to') > 200, pl.col('started_from') <= 1800]))

#df_monthly.select("salary", "started_from", "up_to").unique().head(15)

In [ ]:
# leftover salary converting (not yearly and monthly).

# combine yearly and monthly.
df_combined = pl.concat([df_yearly, df_monthly])

# subtract everything from df_combined, and remove all empty salary rows.
df_left = df.filter(
    ~pl.col("uniq_id").is_in(df_combined["uniq_id"])
)

df_left = df_left.with_columns(
    pl.col("salary").str.strip_chars().alias("salary")
).filter(
    pl.col("salary").is_not_null() & 
    (pl.col("salary") != "")
)

#df_left.select("salary").unique().show

In [ ]:

# Extract single number salary and put it in started_from.

df_left = df_left.with_columns(
    pl.col("salary").str.extract(r"([\d,]+\.?\d*)", 1).str.replace(",", "").alias("salary_single")
)

df_left = df_left.with_columns(
    pl.col("salary_single").cast(pl.Float64, strict=False).alias("salary_single")
)

df_left = df_left.with_columns(
    pl.col("started_from").fill_null(pl.col("salary_single"))
).drop("salary_single")


#df_left.show

In [ ]:
# Hourly salary converting

df_hourly = df_left.filter(pl.col("started_from") < 150)

#df_hourly

In [ ]:
# Convert hourly and monthly into yearly .and round up the decimals.

# Hourly.
df_hourly = df_hourly.with_columns([
    (pl.col("started_from") * 2080).alias("started_from"),
    (pl.col("up_to") * 2080).alias("up_to")
])

# Monthly.
df_monthly = df_monthly.with_columns([
    (pl.col("started_from") * 12).alias("started_from"),
    (pl.col("up_to") * 12).alias("up_to")
])

df_final_salaries = df_final_salaries.with_columns([
    pl.col("started_from").round(2),
    pl.col("up_to").round(2),
    pl.col("average_salary_per_year").round(2)
])

In [ ]:
# Combine everything, remove unrealistic salaries (outliners), and make average column.
df_final_salaries = pl.concat([df_yearly, df_monthly, df_hourly])

df_final_salaries = df_final_salaries.filter(
    (pl.col("up_to") > 4160) &
    (pl.col("up_to") <= 500000)
)

df_final_salaries = df_final_salaries.with_columns(
    pl.when(pl.col("up_to").is_null())
    .then(pl.col("up_to"))
    .otherwise((pl.col("up_to") + pl.col("up_to")) / 2)
    .alias("average_salary_per_year")
)


In [646]:
# Write as csv out.
df_final_salaries.sort(pl.col("average_salary_per_year"), descending=False).write_csv("monster_clean_data.csv")

In [ ]:
# Monster.com Job Listings — Data Cleaning Pipeline

# 1. Load & drop: Remove has_expired (always false) and date_added columns.
# 2. Salary standardization: Salary column contains mixed formats (hourly, monthly, yearly, plain text).
#                            Extract min/max using regex, split into df_yearly, df_monthly, df_hourly,
#                            then convert all into yearly equivalent (hourly x 2080, monthly x 12).
# 3. Outlier removal: Drop rows where salary is unrealistically low (< $4,160) or high (> $500,000).
# 4. Average salary: Create average_salary_per_year as midpoint of min and max.
# 5. Export:Save final cleaned data sorted by average salary as monster_clean_data.csv.
# And we are done!